### Configure API Client

In [ ]:
# Required since it seems cwd is Notebook, rather than where Jupyter was launched from
import sys
sys.path.append('..')

In [ ]:
from src.utils.setup import initialise_binance_client

client = initialise_binance_client(testnet=True)

### Load Data

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../data/raw_ohlcv/BTCUSDT-1h-2017-08-17.csv", index_col='date', parse_dates=['date'])
df = df[["close", "volume"]].copy()

In [ ]:
df.head()

### Calculating Returns

In [ ]:
import numpy as np

In [ ]:
df["return"] = df["close"].div(df["close"].shift(1))     # This is the return factor from that period's interval
df["return"] = np.log(df["return"])    # Converts to log returns for additivity and scale
df

### Visualising Data


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df.close.loc['2021':'2023'].plot(figsize=(12,8))
plt.show()

In [ ]:
df

In [ ]:
fig, axs = plt.subplots(2,2, figsize=(12,8))

df.close.plot(ax=axs[0,0])
df.volume.plot(ax=axs[1,0])
df.close.loc['2021':'2023'].plot(ax=axs[1,1])
df['return'].plot(ax=axs[0,1])

plt.tight_layout()
plt.show()

### Buy and Hold Strategy

In [ ]:
# Normalise Close price with respect to the first price - effectively the multiple at that time
normalised_price_series = df.close / df.close.iloc[0]     

In [ ]:
# Calculating the multiple directly
multiple = np.exp(df["return"].sum())
multiple

In [ ]:
# Cumulative return from buying and holding
df['c_return'] = df['return'].cumsum().apply(np.exp)
df

### Backtesting Random Strategy

#### Backtester

In [ ]:
# Using a subset of data (only 2021) for testing
data = df.loc['2021'].copy()
data['c_return'] = data['return'].cumsum().apply(np.exp)
data

In [ ]:
data['position'] = 1

In [ ]:
# Random strategy - sells when the close price is mod 10
sell_condition = data['close'] % 10 == 0

In [ ]:
data.loc[sell_condition, 'position'] = 0

In [ ]:
# Checks to see how many times the strategy is not holding
data.position.value_counts()

In [ ]:
data['strategy'] = data['position'].shift(1) * data['return']

In [ ]:
data['c_strategy'] = data['strategy'].cumsum().apply(np.exp)

In [ ]:
data[['c_return', 'c_strategy']].plot(figsize=(12,8))
plt.legend(['Buy and Hold', 'Your Strategy'])
plt.show()

#### Performance Metrics

In [ ]:
# Number of trading periods
trading_periods = 8760

##### Annaulised Mean

In [ ]:
ann_mean_log = data[['return', 'strategy']].mean() * trading_periods
ann_mean = np.exp(ann_mean_log) - 1
ann_mean

##### Standard Deviation

In [ ]:
ann_std = data[['return', 'strategy']].std() * np.sqrt(trading_periods)
ann_std

##### Sharpe Ratio

In [ ]:
sharpe = (ann_mean / ann_std)
sharpe

#### Trading Costs

In [ ]:
commission = 0.1 / 100

In [ ]:
ptc = np.log(1 - commission)

In [ ]:
data['trade'] = data['position'].diff().fillna(0).abs()

In [ ]:
data['strategy_net'] = data['strategy'] + data['trade'] * ptc

In [ ]:
data['c_strategy_net'] = data['strategy_net'].cumsum().apply(np.exp)

In [ ]:
data.c_strategy_net.nlargest(10)

In [ ]:
data.c_strategy.nlargest(10)

In [ ]:
data[['c_return', 'c_strategy', 'c_strategy_net']].plot(figsize=(12,8))